# Module 4: Enter the Random Forest

In the previous modules, we saw OLS fail catastrophically and regularization improve Test R² to ~0.5. Now we introduce a fundamentally different approach: the **Random Forest**.

Unlike linear models that estimate one coefficient per feature (and can be fooled by spurious correlations), a Random Forest builds hundreds of decision trees, each seeing only a random subset of features. The result? A model that is **far better at ignoring junk features** while still capturing the signal.

This module is the **reveal** — we will see which features actually matter, which are noise, and why algorithm choice is critical.

In [ ]:
#@title 🔧 Setup { display-mode: "form" }

import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy import stats

from helpers import load_dataset, ROLE_COLORS, apply_dark_theme, takeaway_box, metric_card, role_bar_colors
from helpers.styling import role_legend_html, styled_button, DARK_BG, AXES_BG, TEXT_COLOR, GREEN, RED, GRAY, MUTED_TEXT

X, y, codebook, role_map = load_dataset()

# Shared train/test split used throughout this module
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# StandardScaler for linear models
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

---
## 1. Meet the Random Forest

All the linear models we have tried so far share a common DNA: they estimate **one coefficient per feature** using matrix algebra. When features outnumber samples, the matrix is singular and things break.

The Random Forest takes a completely different approach. Click below to see how it compares.

In [ ]:
#@title 🔧 Meet the Random Forest Widget { display-mode: "form" }

class MeetRandomForestWidget:
    """Trains RF and compares it against all linear models."""

    def __init__(self, X_train, X_test, y_train, y_test, X_train_s, X_test_s):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.X_train_s = X_train_s
        self.X_test_s = X_test_s

        self.train_btn = styled_button('Train Random Forest (500 trees)', width='280px')
        self.train_btn.on_click(self._on_train)
        self.output = widgets.Output()

    def _on_train(self, _):
        with self.output:
            clear_output(wait=True)

            # Train Random Forest on raw (unscaled) features
            rf = RandomForestRegressor(
                n_estimators=500, max_depth=15, random_state=42, n_jobs=-1
            )
            rf.fit(self.X_train, self.y_train)
            rf_train_r2 = r2_score(self.y_train, rf.predict(self.X_train))
            rf_test_r2 = r2_score(self.y_test, rf.predict(self.X_test))

            # RF metric cards
            rf_header = widgets.HTML(f"""
            <h4 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Random Forest Results</h4>
            """)
            rf_cards = widgets.HBox([
                metric_card('RF Train R\u00b2', f'{rf_train_r2:.4f}', color=GREEN, width='220px'),
                metric_card('RF Test R\u00b2', f'{rf_test_r2:.4f}', color=GREEN, width='220px'),
            ])
            display(rf_header, rf_cards)

            # Train all linear models for comparison
            ols = LinearRegression().fit(self.X_train_s, self.y_train)
            ridge = Ridge(alpha=100).fit(self.X_train_s, self.y_train)
            lasso = Lasso(alpha=100, max_iter=5000).fit(self.X_train_s, self.y_train)
            enet = ElasticNet(alpha=10, max_iter=5000).fit(self.X_train_s, self.y_train)

            models = [
                ('OLS', ols),
                ('Ridge (\u03b1=100)', ridge),
                ('Lasso (\u03b1=100)', lasso),
                ('ElasticNet (\u03b1=10)', enet),
            ]

            rows_html = ''
            for name, model in models:
                tr = r2_score(self.y_train, model.predict(self.X_train_s))
                te = r2_score(self.y_test, model.predict(self.X_test_s))
                te_color = GREEN if te > 0.5 else RED
                rows_html += f"""
                <tr style='border-bottom: 1px solid #334155;'>
                    <td style='padding: 10px 14px; color: {MUTED_TEXT};'>{name}</td>
                    <td style='padding: 10px 14px; color: {TEXT_COLOR}; text-align: center;'>{tr:.2f}</td>
                    <td style='padding: 10px 14px; color: {te_color}; text-align: center; font-weight: bold;'>{te:.2f}</td>
                </tr>
                """

            # RF row highlighted
            rows_html += f"""
            <tr style='border: 2px solid #3b82f6; background: #1e3a5f;'>
                <td style='padding: 10px 14px; color: #3b82f6; font-weight: bold;'>Random Forest (500)</td>
                <td style='padding: 10px 14px; color: {TEXT_COLOR}; text-align: center;'>{rf_train_r2:.2f}</td>
                <td style='padding: 10px 14px; color: {GREEN}; text-align: center; font-weight: bold;'>{rf_test_r2:.2f}</td>
            </tr>
            """

            table_html = f"""
            <div style='margin-top: 20px;'>
                <h4 style='color: {TEXT_COLOR}; margin-bottom: 8px;'>Model Comparison</h4>
                <table style='border-collapse: collapse; width: 100%; max-width: 600px;
                              background: #0f172a; border-radius: 10px; overflow: hidden;'>
                    <thead>
                        <tr style='background: #1e293b; border-bottom: 2px solid #475569;'>
                            <th style='padding: 10px 14px; color: {MUTED_TEXT}; text-align: left;'>Model</th>
                            <th style='padding: 10px 14px; color: {MUTED_TEXT}; text-align: center;'>Train R\u00b2</th>
                            <th style='padding: 10px 14px; color: {MUTED_TEXT}; text-align: center;'>Test R\u00b2</th>
                        </tr>
                    </thead>
                    <tbody>{rows_html}</tbody>
                </table>
            </div>
            """
            display(widgets.HTML(table_html))

            summary = widgets.HTML(f"""
            <div style='background: linear-gradient(135deg, #1e3a5f, #1a1a2e);
                        padding: 16px 20px; border-radius: 10px;
                        border-left: 4px solid #3b82f6; margin-top: 15px; max-width: 600px;'>
                <p style='color: {TEXT_COLOR}; font-size: 14px; line-height: 1.6; margin: 0;'>
                    The Random Forest <b>beats every linear model</b> \u2014 even regularized ones.
                    But <b>WHY</b> does it handle 282 features so much better?
                </p>
            </div>
            """)
            display(summary)

    def create_ui(self):
        explanation = widgets.HTML(f"""
        <div style='background: #1e293b; padding: 18px 22px; border-radius: 10px;
                    border: 1px solid #475569; margin-bottom: 15px; max-width: 700px;'>
            <h4 style='color: {TEXT_COLOR}; margin-top: 0;'>How a Random Forest Works</h4>
            <p style='color: {MUTED_TEXT}; font-size: 13px; line-height: 1.7; margin-bottom: 0;'>
                A Random Forest builds <b>hundreds of decision trees</b>, each seeing a
                random subset of features (\u221ap per split) and a random subset of data (bagging).
                It then <b>averages their predictions</b>.<br><br>
                No matrix inversion. No coefficient explosion. Each tree asks simple
                yes/no questions about individual features, and useless features simply
                never get chosen as good split points.
            </p>
        </div>
        """)
        return widgets.VBox([explanation, self.train_btn, self.output])

In [ ]:
#@title ▶️ Launch Meet the Random Forest { display-mode: "form" }

rf_intro = MeetRandomForestWidget(X_train, X_test, y_train, y_test, X_train_s, X_test_s)
display(rf_intro.create_ui())

---
## 2. RF Feature Importance Dashboard

A Random Forest can tell us **which features it actually used**. Each time a feature is used to split a tree node, we can measure how much it reduced prediction error. Summing across all 500 trees gives us a global importance score.

The color coding reveals everything: **green** = causal features (the 22 real economic drivers), **red** = bizarre features (numerology, flag colors, etc.), **gray** = incidental features.

In [ ]:
#@title 🔧 RF Feature Importance Dashboard Widget { display-mode: "form" }

class RFImportanceDashboard:
    """Top-20 RF feature importances, color-coded by role, with summary stats."""

    def __init__(self, X_train, X_test, y_train, y_test, role_map, codebook):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.role_map = role_map
        self.codebook = codebook

        self.show_btn = styled_button('Show Feature Importances', width='240px')
        self.show_btn.on_click(self._on_show)
        self.output = widgets.Output()

    def _on_show(self, _):
        with self.output:
            clear_output(wait=True)

            # Train RF
            rf = RandomForestRegressor(
                n_estimators=500, max_depth=15, random_state=42, n_jobs=-1
            )
            rf.fit(self.X_train, self.y_train)

            # Feature importances
            importances = pd.Series(rf.feature_importances_, index=self.X_train.columns)
            top20 = importances.nlargest(20)

            # Plot
            fig, ax = plt.subplots(figsize=(10, 7))
            apply_dark_theme(fig, ax)

            colors = role_bar_colors(top20.index, self.role_map)
            bars = ax.barh(range(len(top20)), top20.values[::-1], color=colors[::-1], edgecolor='none', height=0.7)
            ax.set_yticks(range(len(top20)))
            ax.set_yticklabels([f.replace('_', ' ') for f in top20.index[::-1]], fontsize=10)
            ax.set_xlabel('Feature Importance', fontsize=12)
            ax.set_title('Top 20 Features by Random Forest Importance', fontsize=14)

            plt.tight_layout()
            plt.show()

            # Role legend
            display(role_legend_html())

            # Compute role-level summary statistics
            total_imp = importances.sum()
            n_features = len(importances)

            role_stats = {}
            for role in ['causal', 'bizarre', 'incidental']:
                feats = [f for f in importances.index if self.role_map.get(f) == role]
                pct_features = len(feats) / n_features * 100
                pct_importance = importances[feats].sum() / total_imp * 100
                role_stats[role] = (len(feats), pct_features, pct_importance)

            causal_n, causal_pf, causal_pi = role_stats['causal']
            bizarre_n, bizarre_pf, bizarre_pi = role_stats['bizarre']
            incidental_n, incidental_pf, incidental_pi = role_stats['incidental']

            cards = widgets.HBox([
                metric_card(
                    f'Causal: {causal_pf:.0f}% of features',
                    f'{causal_pi:.0f}% of importance',
                    color=GREEN, width='240px'
                ),
                metric_card(
                    f'Bizarre: {bizarre_pf:.0f}% of features',
                    f'{bizarre_pi:.1f}% of importance',
                    color=RED, width='240px'
                ),
                metric_card(
                    f'Incidental: {incidental_pf:.0f}% of features',
                    f'{incidental_pi:.0f}% of importance',
                    color=GRAY, width='240px'
                ),
            ])
            display(cards)

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>RF Feature Importance Dashboard</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Click to see which features the Random Forest actually relies on.
        </p>
        """)
        return widgets.VBox([title, self.show_btn, self.output])

In [ ]:
#@title ▶️ Launch RF Feature Importance Dashboard { display-mode: "form" }

rf_dashboard = RFImportanceDashboard(X_train, X_test, y_train, y_test, role_map, codebook)
display(rf_dashboard.create_ui())

---
## 3. Lasso vs RF — Who Gets Fooled?

Lasso is a linear model that performs feature selection by shrinking coefficients to zero. In theory, it should also ignore junk features. But does it?

Let us compare the top-20 features selected by Lasso (by absolute coefficient magnitude) versus the top-20 features used by the Random Forest. The color coding tells the story.

In [ ]:
#@title 🔧 Lasso vs RF Comparison Widget { display-mode: "form" }

class LassoVsRFWidget:
    """Side-by-side comparison of Lasso coefficients vs RF importances."""

    def __init__(self, X_train, X_test, y_train, y_test, X_train_s, X_test_s, role_map):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.X_train_s = X_train_s
        self.X_test_s = X_test_s
        self.role_map = role_map

        self.compare_btn = styled_button('Compare Lasso vs RF', width='220px')
        self.compare_btn.on_click(self._on_compare)
        self.output = widgets.Output()

    def _on_compare(self, _):
        with self.output:
            clear_output(wait=True)

            # Train Lasso (alpha=10 for more selected features)
            lasso = Lasso(alpha=10, max_iter=5000, random_state=42)
            lasso.fit(self.X_train_s, self.y_train)
            lasso_coefs = pd.Series(np.abs(lasso.coef_), index=self.X_train.columns)
            lasso_top20 = lasso_coefs.nlargest(20)

            # Train RF
            rf = RandomForestRegressor(
                n_estimators=500, max_depth=15, random_state=42, n_jobs=-1
            )
            rf.fit(self.X_train, self.y_train)
            rf_importances = pd.Series(rf.feature_importances_, index=self.X_train.columns)
            rf_top20 = rf_importances.nlargest(20)

            # Side-by-side plot
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
            apply_dark_theme(fig, [ax1, ax2])

            # Lasso subplot
            lasso_colors = role_bar_colors(lasso_top20.index, self.role_map)
            ax1.barh(range(len(lasso_top20)), lasso_top20.values[::-1],
                     color=lasso_colors[::-1], edgecolor='none', height=0.7)
            ax1.set_yticks(range(len(lasso_top20)))
            ax1.set_yticklabels([f.replace('_', ' ') for f in lasso_top20.index[::-1]], fontsize=9)
            ax1.set_xlabel('|Coefficient|', fontsize=11)
            ax1.set_title('Lasso (\u03b1=10): Top 20 by |Coefficient|', fontsize=13)

            # RF subplot
            rf_colors = role_bar_colors(rf_top20.index, self.role_map)
            ax2.barh(range(len(rf_top20)), rf_top20.values[::-1],
                     color=rf_colors[::-1], edgecolor='none', height=0.7)
            ax2.set_yticks(range(len(rf_top20)))
            ax2.set_yticklabels([f.replace('_', ' ') for f in rf_top20.index[::-1]], fontsize=9)
            ax2.set_xlabel('Feature Importance', fontsize=11)
            ax2.set_title('Random Forest: Top 20 by Importance', fontsize=13)

            plt.tight_layout()
            plt.show()

            display(role_legend_html())

            # Summary table: role breakdown for each model
            total_lasso = lasso_coefs.sum()
            total_rf = rf_importances.sum()

            summary_rows = ''
            for role, color in [('causal', GREEN), ('bizarre', RED), ('incidental', GRAY)]:
                feats = [f for f in self.X_train.columns if self.role_map.get(f) == role]
                lasso_pct = lasso_coefs[feats].sum() / total_lasso * 100 if total_lasso > 0 else 0
                rf_pct = rf_importances[feats].sum() / total_rf * 100 if total_rf > 0 else 0
                summary_rows += f"""
                <tr style='border-bottom: 1px solid #334155;'>
                    <td style='padding: 10px 14px; color: {color}; font-weight: bold;'>{role.capitalize()}</td>
                    <td style='padding: 10px 14px; color: {TEXT_COLOR}; text-align: center;'>{lasso_pct:.1f}%</td>
                    <td style='padding: 10px 14px; color: {TEXT_COLOR}; text-align: center;'>{rf_pct:.1f}%</td>
                </tr>
                """

            table_html = f"""
            <div style='margin-top: 15px;'>
                <h4 style='color: {TEXT_COLOR}; margin-bottom: 8px;'>Weight Distribution by Role</h4>
                <table style='border-collapse: collapse; width: 100%; max-width: 550px;
                              background: #0f172a; border-radius: 10px; overflow: hidden;'>
                    <thead>
                        <tr style='background: #1e293b; border-bottom: 2px solid #475569;'>
                            <th style='padding: 10px 14px; color: {MUTED_TEXT}; text-align: left;'>Role</th>
                            <th style='padding: 10px 14px; color: {MUTED_TEXT}; text-align: center;'>% of Lasso |coef|</th>
                            <th style='padding: 10px 14px; color: {MUTED_TEXT}; text-align: center;'>% of RF importance</th>
                        </tr>
                    </thead>
                    <tbody>{summary_rows}</tbody>
                </table>
            </div>
            """
            display(widgets.HTML(table_html))

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Lasso vs Random Forest: Who Gets Fooled?</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Compare which features each model relies on. Watch the colors.
        </p>
        """)
        return widgets.VBox([title, self.compare_btn, self.output])

In [ ]:
#@title ▶️ Launch Lasso vs RF Comparison { display-mode: "form" }

lasso_vs_rf = LassoVsRFWidget(X_train, X_test, y_train, y_test, X_train_s, X_test_s, role_map)
display(lasso_vs_rf.create_ui())

---
## 4. Spotlight on Absurdity

Some of our bizarre features are genuinely hilarious. Select one from the dropdown to see how it correlates with GDP, whether Lasso fell for it, and how the Random Forest treated it.

In [ ]:
#@title 🔧 Spotlight on Absurdity Widget { display-mode: "form" }

class SpotlightAbsurdityWidget:
    """Explore individual bizarre features: scatter plot, Lasso selection, RF importance."""

    SPURIOUS_EXPLANATIONS = {
        'name_numerology_score': 'Numerology assigns mystical meaning to numbers derived from letters. Any correlation with GDP is pure coincidence \u2014 the economy does not care about the alphabetical sum of a country name.',
        'scrabble_score_country_name': 'Scrabble tile values were designed for English-language board game balance, not economic forecasting. High-scoring names just happen to share letter patterns with certain wealthy nations.',
        'flag_colors_count': 'The number of colors on a flag reflects heraldic tradition and political history, not economic output. Any correlation is a coincidence of post-colonial design choices.',
        'country_calling_code': 'Calling codes were assigned by the ITU based on geography, not wealth. Europe got low codes early; this accidentally correlates with GDP.',
        'mcdonalds_per_million': 'McDonald\'s expands into wealthy consumer markets, so this is a downstream effect of GDP, not a cause. The burgers did not make the country rich.',
        'capital_scrabble_score': 'Like country name Scrabble scores, capital city scores reflect English letter distributions, not economic fundamentals.',
        'beer_to_wine_ratio': 'Whether a country prefers beer or wine is driven by climate, culture, and colonial history \u2014 not by GDP directly. Correlation is spurious.',
        'name_lucky_number': 'A country\'s lucky number is pseudoscience derived from its name. There is no causal mechanism connecting numerology to economic performance.',
    }

    def __init__(self, X, y, X_train, X_test, y_train, y_test, X_train_s, role_map, codebook):
        self.X = X
        self.y = y
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.X_train_s = X_train_s
        self.role_map = role_map
        self.codebook = codebook

        # Pre-train models
        self.lasso = Lasso(alpha=10, max_iter=5000, random_state=42)
        self.lasso.fit(self.X_train_s, self.y_train)
        self.lasso_coefs = pd.Series(self.lasso.coef_, index=self.X_train.columns)

        self.rf = RandomForestRegressor(
            n_estimators=500, max_depth=15, random_state=42, n_jobs=-1
        )
        self.rf.fit(self.X_train, self.y_train)
        self.rf_importances = pd.Series(self.rf.feature_importances_, index=self.X_train.columns)
        self.rf_ranks = self.rf_importances.rank(ascending=False).astype(int)

        # Build dropdown options (only features that exist in dataset)
        candidate_features = [
            'name_numerology_score', 'scrabble_score_country_name',
            'flag_colors_count', 'country_calling_code',
            'mcdonalds_per_million', 'capital_scrabble_score',
            'beer_to_wine_ratio', 'name_lucky_number',
        ]
        self.feature_options = [f for f in candidate_features if f in self.X.columns]

        self.dropdown = widgets.Dropdown(
            options=[(f.replace('_', ' ').title(), f) for f in self.feature_options],
            value=self.feature_options[0] if self.feature_options else None,
            description='Feature:',
            style={'description_width': '70px'},
            layout=widgets.Layout(width='350px')
        )
        self.dropdown.observe(self._on_select, names='value')

        self.output = widgets.Output()

    def _on_select(self, change):
        feature = change['new']
        if feature is None:
            return
        with self.output:
            clear_output(wait=True)

            x_vals = self.X[feature].values
            y_vals = self.y.values

            # Filter NaN for correlation
            mask = ~(np.isnan(x_vals) | np.isnan(y_vals))
            r_value, p_value = stats.pearsonr(x_vals[mask], y_vals[mask])

            # Scatter plot
            fig, ax = plt.subplots(figsize=(8, 5))
            apply_dark_theme(fig, ax)

            ax.scatter(x_vals[mask], y_vals[mask], c=RED, alpha=0.5, s=30, edgecolors='none')
            # Trend line
            z = np.polyfit(x_vals[mask], y_vals[mask], 1)
            p = np.poly1d(z)
            x_line = np.linspace(np.min(x_vals[mask]), np.max(x_vals[mask]), 100)
            ax.plot(x_line, p(x_line), '--', color='white', alpha=0.7, linewidth=1.5)

            ax.set_xlabel(feature.replace('_', ' ').title(), fontsize=12)
            ax.set_ylabel('GDP per Capita (USD)', fontsize=12)
            ax.set_title(f'{feature.replace("_", " ").title()} vs GDP  (r = {r_value:.3f})', fontsize=13)

            plt.tight_layout()
            plt.show()

            # Description from codebook
            desc_row = self.codebook[self.codebook['column_name'] == feature]
            description = desc_row['description'].values[0] if len(desc_row) > 0 else 'No description available.'

            # Lasso selection
            lasso_selected = self.lasso_coefs[feature] != 0 if feature in self.lasso_coefs.index else False
            lasso_text = f'<span style="color: {RED}; font-weight: bold;">Yes</span>' if lasso_selected else f'<span style="color: {GREEN}; font-weight: bold;">No</span>'

            # RF importance and rank
            rf_imp = self.rf_importances.get(feature, 0)
            rf_rank = int(self.rf_ranks.get(feature, 0))
            n_features = len(self.rf_importances)

            explanation = self.SPURIOUS_EXPLANATIONS.get(feature, 'Any observed correlation is likely spurious \u2014 there is no plausible causal mechanism connecting this feature to GDP.')

            info_html = f"""
            <div style='background: #1e293b; padding: 18px 22px; border-radius: 10px;
                        border: 1px solid #475569; margin-top: 10px; max-width: 700px;'>
                <p style='color: {MUTED_TEXT}; font-size: 13px; margin: 4px 0;'>
                    <b style='color: {TEXT_COLOR};'>Description:</b> {description}
                </p>
                <p style='color: {MUTED_TEXT}; font-size: 13px; margin: 4px 0;'>
                    <b style='color: {TEXT_COLOR};'>Lasso selected this feature:</b> {lasso_text}
                </p>
                <p style='color: {MUTED_TEXT}; font-size: 13px; margin: 4px 0;'>
                    <b style='color: {TEXT_COLOR};'>RF importance:</b> {rf_imp:.4f} (rank #{rf_rank} out of {n_features})
                </p>
                <hr style='border-color: #334155; margin: 10px 0;'>
                <p style='color: {MUTED_TEXT}; font-size: 13px; font-style: italic; margin: 4px 0;'>
                    {explanation}
                </p>
            </div>
            """
            display(widgets.HTML(info_html))

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Spotlight on Absurdity</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Select a bizarre feature to explore its relationship with GDP.
        </p>
        """)
        return widgets.VBox([title, self.dropdown, self.output])

In [ ]:
#@title ▶️ Launch Spotlight on Absurdity { display-mode: "form" }

spotlight = SpotlightAbsurdityWidget(X, y, X_train, X_test, y_train, y_test, X_train_s, role_map, codebook)
display(spotlight.create_ui())

---
## 5. Build Your Own Model

Now it is your turn. Choose which features to include and which algorithm to use. The key insight: **the Random Forest barely changes when you remove junk features**, while OLS collapses when you add them.

In [ ]:
#@title 🔧 Build Your Own Model Widget { display-mode: "form" }

class BuildYourOwnModelWidget:
    """Interactive widget to train models on different feature subsets."""

    INSIGHTS = {
        ('causal', 'Random Forest'): 'The Random Forest with just 22 causal features performs nearly as well as with all 282 features. It was already ignoring the junk!',
        ('causal', 'OLS'): 'OLS with only causal features works much better than with all 282. Removing the noise fixed the overfitting problem \u2014 but you needed domain expertise to know which features to keep.',
        ('causal', 'Ridge (\u03b1=100)'): 'Ridge with causal features performs well. Regularization + good feature selection is a solid combination.',
        ('causal', 'Lasso (\u03b1=100)'): 'Lasso with causal features performs reasonably. With fewer features, there is less junk to accidentally select.',
        ('bizarre', 'Random Forest'): 'Even with only bizarre features, RF gets a positive R\u00b2. Some bizarre features (like McDonald\'s count) genuinely correlate with GDP \u2014 but correlation is not causation.',
        ('bizarre', 'OLS'): 'OLS with 92 bizarre features and only ~200 training samples \u2014 the system is under-determined again. Expect wild overfitting.',
        ('bizarre', 'Ridge (\u03b1=100)'): 'Ridge can extract some signal even from bizarre features \u2014 several of them do correlate with GDP, even if spuriously.',
        ('bizarre', 'Lasso (\u03b1=100)'): 'Lasso selects a handful of bizarre features that happen to correlate with GDP. This is exactly how spurious models get built in practice.',
        ('all', 'Random Forest'): 'RF with all 282 features performs about the same as with just 22 causal ones. The forest naturally concentrates on the best predictors.',
        ('all', 'OLS'): 'OLS with all 282 features: classic overfitting. Perfect training fit, disastrous test performance.',
        ('all', 'Ridge (\u03b1=100)'): 'Ridge with all features: regularization helps, but the model still spreads weight across bizarre features.',
        ('all', 'Lasso (\u03b1=100)'): 'Lasso with all 282 features selects a mix of causal and bizarre features. Feature selection is better than OLS, but worse than RF importance.',
    }

    def __init__(self, X, y, role_map):
        self.X = X
        self.y = y
        self.role_map = role_map

        # Precompute feature subsets
        self.causal_feats = [f for f in X.columns if role_map.get(f) == 'causal']
        self.bizarre_feats = [f for f in X.columns if role_map.get(f) == 'bizarre']
        self.all_feats = list(X.columns)

        self.feature_toggle = widgets.ToggleButtons(
            options=[
                (f'Only Causal ({len(self.causal_feats)})', 'causal'),
                (f'Only Bizarre ({len(self.bizarre_feats)})', 'bizarre'),
                (f'All Features ({len(self.all_feats)})', 'all'),
            ],
            value='causal',
            description='Features:',
            style={'description_width': '80px', 'button_width': '180px'},
        )

        self.model_dropdown = widgets.Dropdown(
            options=['OLS', 'Ridge (\u03b1=100)', 'Lasso (\u03b1=100)', 'Random Forest'],
            value='Random Forest',
            description='Model:',
            style={'description_width': '60px'},
            layout=widgets.Layout(width='300px')
        )

        self.train_btn = styled_button('Train & Compare', width='180px')
        self.train_btn.on_click(self._on_train)
        self.output = widgets.Output()

    def _on_train(self, _):
        with self.output:
            clear_output(wait=True)

            feat_choice = self.feature_toggle.value
            model_name = self.model_dropdown.value

            if feat_choice == 'causal':
                feats = self.causal_feats
            elif feat_choice == 'bizarre':
                feats = self.bizarre_feats
            else:
                feats = self.all_feats

            X_sub = self.X[feats]
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_sub, self.y, test_size=0.2, random_state=42
            )

            # Choose and train model
            if model_name == 'Random Forest':
                model = RandomForestRegressor(
                    n_estimators=500, max_depth=15, random_state=42, n_jobs=-1
                )
                model.fit(X_tr, y_tr)
                train_r2 = r2_score(y_tr, model.predict(X_tr))
                test_r2 = r2_score(y_te, model.predict(X_te))
            else:
                sc = StandardScaler()
                X_tr_s = sc.fit_transform(X_tr)
                X_te_s = sc.transform(X_te)

                if model_name == 'OLS':
                    model = LinearRegression()
                elif model_name == 'Ridge (\u03b1=100)':
                    model = Ridge(alpha=100)
                else:
                    model = Lasso(alpha=100, max_iter=5000)

                model.fit(X_tr_s, y_tr)
                train_r2 = r2_score(y_tr, model.predict(X_tr_s))
                test_r2 = r2_score(y_te, model.predict(X_te_s))

            train_color = GREEN if train_r2 > 0.5 else RED
            test_color = GREEN if test_r2 > 0.4 else RED

            cards = widgets.HBox([
                metric_card('Train R\u00b2', f'{train_r2:.4f}', color=train_color, width='220px'),
                metric_card('Test R\u00b2', f'{test_r2:.4f}', color=test_color, width='220px'),
            ])

            info = widgets.HTML(f"""
            <div style='color: {MUTED_TEXT}; font-size: 12px; margin-top: 5px;'>
                Features: {len(feats)} &nbsp;|&nbsp; Model: {model_name}
                &nbsp;|&nbsp; Training samples: {len(X_tr)} &nbsp;|&nbsp; Test samples: {len(X_te)}
            </div>
            """)

            # Insight text
            insight_key = (feat_choice, model_name)
            insight_text = self.INSIGHTS.get(insight_key, 'Try different combinations to discover how feature selection and model choice interact.')

            insight = widgets.HTML(f"""
            <div style='background: linear-gradient(135deg, #1e3a5f, #1a1a2e);
                        padding: 14px 18px; border-radius: 10px;
                        border-left: 4px solid #3b82f6; margin-top: 12px; max-width: 650px;'>
                <p style='color: {TEXT_COLOR}; font-size: 13px; line-height: 1.6; margin: 0;'>
                    {insight_text}
                </p>
            </div>
            """)

            display(cards, info, insight)

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Build Your Own Model</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Choose a feature subset and a model, then click <b>Train &amp; Compare</b>.
        </p>
        """)
        controls = widgets.VBox([
            self.feature_toggle,
            self.model_dropdown,
            self.train_btn
        ])
        return widgets.VBox([title, controls, self.output])

In [ ]:
#@title ▶️ Launch Build Your Own Model { display-mode: "form" }

build_model = BuildYourOwnModelWidget(X, y, role_map)
display(build_model.create_ui())

---
## Takeaway

In [ ]:
#@title 🔧 Takeaway { display-mode: "form" }

display(takeaway_box(
    "Random Forest outperforms all linear models (R\u00b2\u22480.61 vs 0.51) <b>AND</b> concentrates "
    "over 50% of its importance on just 22 causal features. Lasso got fooled by numerology "
    "and flag colors. Different algorithms have different failure modes \u2014 but <b>none can "
    "prove causation</b>. Domain expertise is irreplaceable."
))